In [42]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

from datetime import datetime, timedelta
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import tiktoken
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser



In [43]:
# Load and process documents
directory_loader = DirectoryLoader("data", glob="**/*.pdf", loader_cls=PyMuPDFLoader)
student_health_resources = directory_loader.load()

def tiktoken_len(text):
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(text)
    return len(tokens)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750,
    chunk_overlap=0,
    length_function=tiktoken_len,
)

student_health_chunks = text_splitter.split_documents(student_health_resources)

# Setup vectorstore
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
qdrant_vectorstore = Qdrant.from_documents(
    documents=student_health_chunks,
    embedding=embedding_model,
    location=":memory:"
)
qdrant_retriever = qdrant_vectorstore.as_retriever()

# RAG prompt and model
HUMAN_TEMPLATE = """
#CONTEXT:
{context}

QUERY:
{query}

Use the provided context to answer the user query about student health, wellness, nutrition, stress management, 
sleep, exercise, mental health, or any student success topics. Only use the provided context to answer the query. 
If you do not know the answer, or it's not contained in the provided context respond with "I don't know"
"""

chat_prompt = ChatPromptTemplate.from_messages([
    ("human", HUMAN_TEMPLATE)
])

rag_model = ChatOpenAI(model="gpt-4.1-nano")


In [14]:
# LCEL chain
naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"query" : "<<SOME USER QUESTION>>"}
    # "query" : populated by getting the value of the "query" key
    # "context"  : populated by getting the value of the "query" key and chaining it into the base_retriever
    {"context": itemgetter("query") | qdrant_retriever, "query": itemgetter("query")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "query" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": chat_prompt | rag_model, "context": itemgetter("context")}
)

# naive_retrieval_chain.invoke({"query" : "What snacks should I stock up in my dorm room fridge and why?"})["response"].content

In [ ]:
# Creating golden dataset with RAGAS

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(student_health_resources[:20], testset_size=10)

dataset.to_pandas()

Applying HeadlinesExtractor:   0%|          | 0/5 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/8 [00:00<?, ?it/s]

Property 'summary' already exists in node '686011'. Skipping!
Property 'summary' already exists in node '302969'. Skipping!
Property 'summary' already exists in node 'a981cc'. Skipping!


...

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/9 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,Wut can students do at the Unversity of Oregon...,[Exercises to Try Exercise can happen nearly e...,"At the University of Oregon, students can enga...",single_hop_specifc_query_synthesizer
1,How can college students incorporate exercise ...,[Exercises to Try Exercise can happen nearly e...,"According to Shape's recommendations, college ...",single_hop_specifc_query_synthesizer
2,How can participating in Bagua classes contrib...,[Off Campus - Hiking: Use free mornings to esc...,Participating in Bagua classes can contribute ...,single_hop_specifc_query_synthesizer
3,What is Bagua in the context of fitness option...,[Off Campus - Hiking: Use free mornings to esc...,Bagua is an uncommon martial art that can be e...,single_hop_specifc_query_synthesizer
4,What are some effective ways for a busy colleg...,[<1-hop>\n\nExercises to Try Exercise can happ...,A busy college student can incorporate exercis...,multi_hop_abstract_query_synthesizer
5,What are some effective ways to incorporate ex...,[<1-hop>\n\nExercises to Try Exercise can happ...,To incorporate exercise into a busy college sc...,multi_hop_abstract_query_synthesizer
6,What are some exercises that can improve cardi...,[<1-hop>\n\nExercises to Try Exercise can happ...,Exercises that can improve cardiovascular heal...,multi_hop_abstract_query_synthesizer
7,How can hiking and gym classes help with exerc...,[<1-hop>\n\nExercises to Try Exercise can happ...,Hiking can help busy college students by provi...,multi_hop_abstract_query_synthesizer
8,How can a busy college student incorporate hik...,[<1-hop>\n\nExercises to Try Exercise can happ...,A busy college student can incorporate hiking ...,multi_hop_abstract_query_synthesizer


In [16]:
# Evaluating RAG system with RAGAS with the golden dataset made with SDG

from ragas import EvaluationDataset
from ragas import evaluate
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity, ContextPrecision
from ragas import evaluate, RunConfig

for test_row in dataset:
  response = naive_retrieval_chain.invoke({"query" : test_row.eval_sample.user_input}) ## input with given retriever
  test_row.eval_sample.response = response["response"].content
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]



evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())



evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))



custom_run_config = RunConfig(timeout=360)

result = evaluate(
    dataset=evaluation_dataset,
    metrics=[Faithfulness(), FactualCorrectness(), ResponseRelevancy(), LLMContextRecall(), ContextEntityRecall(), NoiseSensitivity(), ContextPrecision()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
result

Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

{'faithfulness': 0.9071, 'factual_correctness(mode=f1)': 0.5044, 'answer_relevancy': 0.9706, 'context_recall': 0.9352, 'context_entity_recall': 0.2072, 'noise_sensitivity(mode=relevant)': 0.4790, 'context_precision': 1.0000}

In [17]:
# Contextual compression retriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=qdrant_retriever
)

cohere_retrieval_chain = (
    {"context": itemgetter("query") | compression_retriever, "query": itemgetter("query")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": chat_prompt | rag_model, "context": itemgetter("context")}
)

# cohere_retrieval_chain.invoke({"query" : "What are some quick exercises I can do?"})["response"].content

In [18]:
# Multi-query retriever
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=qdrant_retriever, llm=rag_model
)

multi_query_retrieval_chain = (
    {"context": itemgetter("query") | multi_query_retriever, "query": itemgetter("query")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": chat_prompt | rag_model, "context": itemgetter("context")}
)

# multi_query_retrieval_chain.invoke({"query" : "I just took an exam and I'm feeling nervous. How do I calm down?"})["response"].content

In [19]:
# Parent document retriever
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = student_health_resources
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

parent_document_retriever.add_documents(parent_docs, ids=None)

parent_document_retrieval_chain = (
    {"context": itemgetter("query") | parent_document_retriever, "query": itemgetter("query")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": chat_prompt | rag_model, "context": itemgetter("context")}
)

# parent_document_retrieval_chain.invoke({"query" : "Why is bad nutrition a problem?"})["response"].content

In [ ]:
# Ensemble retriever
from langchain.retrievers import EnsembleRetriever

retriever_list = [qdrant_retriever, compression_retriever, multi_query_retriever, parent_document_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

ensemble_retrieval_chain = (
    {"context": itemgetter("query") | ensemble_retriever, "query": itemgetter("query")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": chat_prompt | rag_model, "context": itemgetter("context")}
)

# ensemble_retrieval_chain.invoke({"query" : "I heard that sleeping 4 hours a day is fine. Is that true?"})["response"].content

In [ ]:
# Evaluating all retrievers with metrics, using RAGAS
from ragas import RunConfig
import time


def evaluate_all_retrievers_with_metrics(synthetic_dataset):
    """Evaluate all retrievers using the same synthetic dataset and evaluator"""

    encoder = tiktoken.encoding_for_model("gpt-4o-mini")
    
    retrievers = {
        "naive": naive_retrieval_chain,
        "multi_query": multi_query_retrieval_chain,
        "parent_document": parent_document_retrieval_chain,
        "compression": cohere_retrieval_chain,
        "ensemble": ensemble_retrieval_chain
    }
    
    results = {}
    
    for name, retriever_chain in retrievers.items():
        print(f"Evaluating {name} retriever...")
        
        # Measure latency and cost
        latencies = []
        costs = []
        
        # Process each test case and collect metrics
        for test_row in synthetic_dataset:
            print("now measuring latency") # Measure latency
            start_time = time.time()
            response = retriever_chain.invoke({"query": test_row.eval_sample.user_input})
            end_time = time.time()
            latencies.append(end_time - start_time)
            
            print("now estimating cost") # Estimate cost
            # Use the encoder to count tokens accurately
            input_text = test_row.eval_sample.user_input + " ".join([context.page_content for context in response["context"]])
            output_text = response["response"].content

            input_tokens = len(encoder.encode(input_text))
            output_tokens = len(encoder.encode(output_text))

            input_cost = input_tokens * 0.00000015  # $0.15 per 1M tokens
            output_cost = output_tokens * 0.00000060  # $0.60 per 1M tokens
            total_cost = input_cost + output_cost

            costs.append(total_cost)
            
            # Update dataset for Ragas evaluation (same as your existing code)
            test_row.eval_sample.response = response["response"].content
            test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
        
        # Calculate latency and cost metrics
        latency_metrics = {
            "avg_latency_seconds": sum(latencies) / len(latencies),
            "min_latency_seconds": min(latencies),
            "max_latency_seconds": max(latencies)
        }
        
        cost_metrics = {
            "avg_cost_usd": sum(costs) / len(costs),
            "total_cost_usd": sum(costs),
            "min_cost_usd": min(costs),
            "max_cost_usd": max(costs)
        }
        
        # Get Ragas metrics (using the same evaluator_llm)
        evaluation_dataset = EvaluationDataset.from_pandas(synthetic_dataset.to_pandas())
        evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))
        custom_run_config = RunConfig(timeout=360)
        
        print("now in the evaluation phase")
        ragas_result = evaluate(
            dataset=evaluation_dataset,
            metrics=[Faithfulness(), FactualCorrectness(), ResponseRelevancy(), LLMContextRecall(), ContextEntityRecall(), NoiseSensitivity(), ContextPrecision()],
            llm=evaluator_llm,  # Same evaluator for all
            run_config=custom_run_config
        )
        
        results[name] = {
            "latency_metrics": latency_metrics,
            "cost_metrics": cost_metrics,
            "ragas_metrics": ragas_result
        }
    
    return results

results_retrievers = evaluate_all_retrievers_with_metrics(dataset)

Evaluating naive retriever...
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now in the evaluation phase


Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

Evaluating multi_query retriever...
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now in the evaluation phase


Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

Evaluating parent_document retriever...
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now in the evaluation phase


Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

Evaluating compression retriever...
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now in the evaluation phase


Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

Evaluating ensemble retriever...
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now measuring latency
now estimating cost
now in the evaluation phase


Evaluating:   0%|          | 0/63 [00:00<?, ?it/s]

In [34]:
results_retrievers

{'naive': {'latency_metrics': {'avg_latency_seconds': 3.4787993166181774,
   'min_latency_seconds': 1.2874269485473633,
   'max_latency_seconds': 12.687006950378418},
  'cost_metrics': {'avg_cost_usd': 0.0003700166666666666,
   'total_cost_usd': 0.0033301499999999996,
   'min_cost_usd': 0.0003138,
   'max_cost_usd': 0.00047459999999999994},
  'ragas_metrics': {'faithfulness': 0.9740, 'factual_correctness(mode=f1)': 0.5456, 'answer_relevancy': 0.9658, 'context_recall': 0.9500, 'context_entity_recall': 0.2067, 'noise_sensitivity(mode=relevant)': 0.3953, 'context_precision': 1.0000}},
 'multi_query': {'latency_metrics': {'avg_latency_seconds': 5.137515041563246,
   'min_latency_seconds': 3.5907630920410156,
   'max_latency_seconds': 7.346039056777954},
  'cost_metrics': {'avg_cost_usd': 0.0003612333333333333,
   'total_cost_usd': 0.0032511,
   'min_cost_usd': 0.00031094999999999994,
   'max_cost_usd': 0.0003948},
  'ragas_metrics': {'faithfulness': 0.9127, 'factual_correctness(mode=f1)': 

In [40]:
import pandas as pd

# Manually constructed data because the result from the last code cell created a host of errors when creating the data frame
comparison_data = {
    'Metric': [
        'Avg Latency (s)',
        'Avg Cost ($)',
        'Faithfulness',
        'Factual Correctness',
        'Answer Relevancy',
        'Context Recall',
        'Context Entity Recall',
        'Noise Sensitivity',
        'Context Precision'
    ],
    'NAIVE': [
        3.4788,
        0.000370,
        0.9740,
        0.5456,
        0.9658,
        0.9500,
        0.2067,
        0.3953,
        1.0000
    ],
    'MULTI_QUERY': [
        5.1375,
        0.000361,
        0.9127,
        0.5378,
        0.9643,
        0.9722,
        0.1234,
        0.3698,
        0.8611
    ],
    'PARENT_DOCUMENT': [
        2.0118,
        0.000338,
        0.8217,
        0.5533,
        0.8587,
        0.9722,
        0.2188,
        0.2840,
        0.9444
    ],
    'COMPRESSION': [
        3.1267,
        0.000303,
        0.9389,
        0.6078,
        0.9603,
        0.9352,
        0.1697,
        0.2894,
        0.9444
    ],
    'ENSEMBLE': [
        6.8106,
        0.000605,
        0.9861,
        0.4833,
        0.9752,
        0.9722,
        0.1877,
        0.4957,
        0.8015
    ]
}

# Create DataFrame and print
comparison_df = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("SIDE-BY-SIDE RETRIEVER COMPARISON")
print("="*80)
print(comparison_df.to_string(index=False))


SIDE-BY-SIDE RETRIEVER COMPARISON
               Metric   NAIVE  MULTI_QUERY  PARENT_DOCUMENT  COMPRESSION  ENSEMBLE
      Avg Latency (s) 3.47880     5.137500         2.011800     3.126700  6.810600
         Avg Cost ($) 0.00037     0.000361         0.000338     0.000303  0.000605
         Faithfulness 0.97400     0.912700         0.821700     0.938900  0.986100
  Factual Correctness 0.54560     0.537800         0.553300     0.607800  0.483300
     Answer Relevancy 0.96580     0.964300         0.858700     0.960300  0.975200
       Context Recall 0.95000     0.972200         0.972200     0.935200  0.972200
Context Entity Recall 0.20670     0.123400         0.218800     0.169700  0.187700
    Noise Sensitivity 0.39530     0.369800         0.284000     0.289400  0.495700
    Context Precision 1.00000     0.861100         0.944400     0.944400  0.801500
